In [ ]:
%matplotlib qt
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, RadioButtons
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from torch.amp import autocast

from PGI_Hnet import PhysicsGuidedInversionUNet
from nanothermal_3D_lib import Native3DTQSISolver, MATERIALS_DB

VERSION = input("Enter the trained model version (e.g., 7.8): ").strip()
MODEL_PATH = rf"D:\kushagra\experimental data\simulation data loader\PGI_Hnet checkpoints\pinn_latest_V{VERSION}.pth"

# =========================================================================
# --- GLOBAL PARAMETERS ---
# =========================================================================
NX, NY, NZ_MODEL = 128, 128, 32
WINDOW_XY_UM = 12.8  
THICKNESS_Z_UM = 200.0 
PIXEL_SIZE_UM = WINDOW_XY_UM / NX
Z_ARRAY_UM = np.linspace(0, THICKNESS_Z_UM, NZ_MODEL)

P_PUMP_W = 15.0e-3  
LAMBDA_PUMP_NM = 532.0
LAMBDA_PROBE_NM = 457.0  
LAYER_MATERIALS = ['glass', 'water', 'glass']
NP_MATERIAL = 'gold'    

# =========================================================================
# --- COMPLEX PATTERN & PHYSICS GENERATOR ---
# =========================================================================
def interp_pts(p1, p2, num, r):
    """Interpolate coordinates to draw lines with a specific particle radius."""
    return [(x, y, r) for x, y in zip(np.linspace(p1[0], p2[0], num), np.linspace(p1[1], p2[1], num))]

# 1. Sigma Symbol (Σ)
pts_sigma = interp_pts((2.5, 3.5), (-2.5, 3.5), 6, 60)[:-1] + \
            interp_pts((-2.5, 3.5), (1.5, 0), 5, 45)[:-1] + \
            interp_pts((1.5, 0), (-2.5, -3.5), 5, 45)[:-1] + \
            interp_pts((-2.5, -3.5), (2.5, -3.5), 6, 30)

# 2. Pi Symbol (Π)
pts_pi = interp_pts((-3.5, 3), (3.5, 3), 8, 50) + \
         interp_pts((-2.0, 2), (-2.0, -3), 5, 35) + \
         interp_pts((2.0, 2), (2.0, -3), 5, 35)

# 3. Archimedean Spiral (Growing sizes)
pts_spiral = []
for i in range(22):
    theta = i * (3 * np.pi / 18)
    rad_pos = 0.25 * theta
    r_np = 20 + i * 2.5  # Particles grow from 20nm to ~75nm outward
    pts_spiral.append((rad_pos * np.cos(theta), rad_pos * np.sin(theta), r_np))

# 4. Hexagonal Flower
pts_flower = [(0, 0, 80)] # Giant center
for i in range(6): pts_flower.append((1.5*np.cos(i*np.pi/3), 1.5*np.sin(i*np.pi/3), 30))
for i in range(12): pts_flower.append((3.5*np.cos(i*np.pi/6), 3.5*np.sin(i*np.pi/6), 50))

# 5. Chaos Scatter
np.random.seed(42)
pts_chaos = [(np.random.uniform(-4.5, 4.5), np.random.uniform(-4.5, 4.5), np.random.uniform(20, 80)) for _ in range(18)]

# Master Configuration Dictionary
PATTERN_CONFIGS = {
    'Sigma_L1': {
        'pts': pts_sigma, 'b1': 70.0, 'b2': 140.0, 'z0': 35.0,  # Z in Layer 1
        'k_mid': 0.6, 'desc': "Σ symbol in Bottom Glass"
    },
    'Pi_L3': {
        'pts': pts_pi, 'b1': 40.0, 'b2': 120.0, 'z0': 160.0,    # Z in Layer 3
        'k_mid': 0.8, 'desc': "Π symbol in Top Glass (High K fluid)"
    },
    'Spiral_Mid': {
        'pts': pts_spiral, 'b1': 60.0, 'b2': 150.0, 'z0': 105.0, # Z in Layer 2
        'k_mid': 0.2, 'desc': "Growing Spiral in Insulating Fluid"
    },
    'Flower_Thin': {
        'pts': pts_flower, 'b1': 90.0, 'b2': 110.0, 'z0': 100.0, # Ultra-thin fluid gap
        'k_mid': 0.6, 'desc': "Hex Flower in 20µm thin water channel"
    },
    'Chaos_Interface': {
        'pts': pts_chaos, 'b1': 50.0, 'b2': 150.0, 'z0': 148.0,  # Right at the interface
        'k_mid': 0.6, 'desc': "Random cluster at Top Glass Interface"
    }
}
PATTERNS = list(PATTERN_CONFIGS.keys())

# =========================================================================
# --- SIMULATION & MODEL INFERENCE ---
# =========================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Loading Model Version {VERSION} onto {device}...")
model = PhysicsGuidedInversionUNet(bf=8).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True), strict=False)
model.eval()

def compute_case(pat_name):
    print(f"    -> Simulating '{pat_name}'...")
    cfg = PATTERN_CONFIGS[pat_name]
    
    # Unpack specific physics for this pattern
    b1_um, b2_um, z0_um, k_mid = cfg['b1'], cfg['b2'], cfg['z0'], cfg['k_mid']
    
    solver = Native3DTQSISolver(
        pixel_size_um=PIXEL_SIZE_UM, z_array_um=Z_ARRAY_UM,
        layer_materials=LAYER_MATERIALS, boundaries_um=[b1_um, b2_um],
        a_m=50e-9, probe_nm=LAMBDA_PROBE_NM
    )
    
    # Dynamically determine the local refractive index based on Z placement
    if z0_um < b1_um or z0_um > b2_um:
        local_n = MATERIALS_DB['glass']['n']
    else:
        local_n = MATERIALS_DB['water']['n']

    particles_array = []
    for x, y, r in cfg['pts']:
        # Recalculate sigma_abs uniquely for EVERY particle size and local environment
        sig = Native3DTQSISolver.compute_optical_properties(r, LAMBDA_PUMP_NM, MATERIALS_DB[NP_MATERIAL]['n'], local_n)
        particles_array.append({'x0': x, 'y0': y, 'radius_nm': r, 'material': NP_MATERIAL, 'sigma_abs': sig})
    
    _, T_true_3d, source_3d = solver.generate_synthetic_phase(NY, NX, particles_array, P_PUMP_W, LAMBDA_PUMP_NM, z0_um=z0_um)
    
    T_tensor = torch.tensor(T_true_3d, dtype=solver.dtype, device=solver.device)
    dn_3d = torch.zeros_like(T_tensor)
    
    m_bot = (solver.z_array < solver.b1).view(-1,1,1).expand_as(dn_3d)
    m_mid = ((solver.z_array >= solver.b1) & (solver.z_array <= solver.b2)).view(-1,1,1).expand_as(dn_3d)
    m_top = (solver.z_array > solver.b2).view(-1,1,1).expand_as(dn_3d)
    
    dn_3d[m_bot] = solver._eval_dn(T_tensor, solver.beta_bot)[m_bot]
    dn_3d[m_mid] = solver._eval_dn(T_tensor, solver.beta_mid)[m_mid]
    dn_3d[m_top] = solver._eval_dn(T_tensor, solver.beta_top)[m_top] 
    
    opd_2d = torch.sum(dn_3d * solver.dz, dim=0)
    phase_2d = (opd_2d * (2 * np.pi / solver.lambda_probe)).cpu().numpy()
    _, T_inv_3d = solver.invert_tqsi_3d(phase_2d, z0_um=z0_um, iterations=3)
    
    # Build 3D domain tensors with custom geometry
    k_vol = np.zeros((NZ_MODEL, NY, NX), dtype=np.float32)
    beta_vol = np.zeros((NZ_MODEL, NY, NX), dtype=np.float32)
    k_vol[Z_ARRAY_UM < b1_um] = MATERIALS_DB['glass']['k']
    k_vol[(Z_ARRAY_UM >= b1_um) & (Z_ARRAY_UM <= b2_um)] = k_mid
    k_vol[Z_ARRAY_UM > b2_um] = MATERIALS_DB['glass']['k']
    
    beta_vol[Z_ARRAY_UM < b1_um] = MATERIALS_DB['glass']['beta'][0]
    beta_vol[(Z_ARRAY_UM >= b1_um) & (Z_ARRAY_UM <= b2_um)] = MATERIALS_DB['water']['beta'][0]
    beta_vol[Z_ARRAY_UM > b2_um] = MATERIALS_DB['glass']['beta'][0]

    with torch.no_grad():
        with autocast('cuda'):
            phase_t = torch.tensor(phase_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            t_inv_t = torch.tensor(T_inv_3d, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            k_t     = torch.tensor(k_vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            beta_t  = torch.tensor(beta_vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            
            T_pred_tensor, _ = model(phase_t, t_inv_t, k_t, beta_t)
            T_pred_3d = T_pred_tensor.squeeze().cpu().numpy()

    error_map = np.abs(T_pred_3d - T_true_3d) / (np.max(T_true_3d) + 1e-5) * 100.0

    SMOOTH_NZ = 96 
    smooth_z_array = np.linspace(0, THICKNESS_Z_UM, SMOOTH_NZ)
    T_true_3d = interp1d(Z_ARRAY_UM, T_true_3d, axis=0, kind='cubic')(smooth_z_array)
    T_pred_3d = interp1d(Z_ARRAY_UM, T_pred_3d, axis=0, kind='cubic')(smooth_z_array)
    error_map = interp1d(Z_ARRAY_UM, error_map, axis=0, kind='linear')(smooth_z_array)

    return {
        'phase': phase_2d, 'T_true': T_true_3d, 'T_pred': T_pred_3d, 
        'error': error_map, 'source': source_3d, 'z_arr': smooth_z_array
    }

print("\n[*] PRE-COMPUTING EXTREME OOD SCENARIOS (This will take ~10 seconds)...")
results_db = {pat: compute_case(pat) for pat in PATTERNS}
print("[*] Pre-computation complete! Launching interactive dashboard.")

# =========================================================================
# --- GUI SETUP ---
# =========================================================================
active_pat = PATTERNS[0]
data = results_db[active_pat]
Z_ARRAY_SMOOTH = data['z_arr']
NZ = len(Z_ARRAY_SMOOTH)

y_ax = np.linspace(-NY//2, NY//2 - 1, NY) * PIXEL_SIZE_UM
x_ax = np.linspace(-NX//2, NX//2 - 1, NX) * PIXEL_SIZE_UM

cfg = PATTERN_CONFIGS[active_pat]
z_center = np.argmin(np.abs(Z_ARRAY_SMOOTH - cfg['z0'])) 
y_center, x_center = NY // 2, NX // 2

fig = plt.figure(figsize=(24, 12)) 
gs = GridSpec(2, 5, figure=fig, height_ratios=[1.5, 1], width_ratios=[1, 1, 1, 1, 1.2], 
              bottom=0.25, top=0.92, hspace=0.3, wspace=0.3) 

main_title = fig.suptitle(f"PINN Generalization Testing | OOD Complex Patterns V{VERSION}", fontsize=18, weight='bold', y=0.98)

ax_phase = fig.add_subplot(gs[0, 0])
im_phase = ax_phase.imshow(data['phase'], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='viridis')
ax_phase.set_title("Input 2D Phase Map (rad)")
cb_phase = plt.colorbar(im_phase, ax=ax_phase, fraction=0.046, pad=0.04)

vmax_shared = max(np.max(data['T_true'][z_center]), np.max(data['T_pred'][z_center])) + 1e-5

ax_true = fig.add_subplot(gs[0, 1])
im_true = ax_true.imshow(data['T_true'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='hot', vmin=0, vmax=vmax_shared)
ax_true.set_title(r"Ground Truth $T_{true}(x,y)$")
cb_true = plt.colorbar(im_true, ax=ax_true, fraction=0.046, pad=0.04)
hl_true = ax_true.axhline(y_ax[y_center], color='cyan', linestyle='--', alpha=0.7)
vl_true = ax_true.axvline(x_ax[x_center], color='lime', linestyle='--', alpha=0.7)

ax_pred = fig.add_subplot(gs[0, 2])
im_pred = ax_pred.imshow(data['T_pred'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='hot', vmin=0, vmax=vmax_shared)
ax_pred.set_title(r"PINN Prediction $T_{pred}(x,y)$")
cb_pred = plt.colorbar(im_pred, ax=ax_pred, fraction=0.046, pad=0.04)
hl_pred = ax_pred.axhline(y_ax[y_center], color='cyan', linestyle='--', alpha=0.7)
vl_pred = ax_pred.axvline(x_ax[x_center], color='lime', linestyle='--', alpha=0.7)

ax_err = fig.add_subplot(gs[0, 3])
im_err = ax_err.imshow(data['error'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='Reds', vmin=0, vmax=20)
ax_err.set_title("Absolute Error (%)")
cb_err = plt.colorbar(im_err, ax=ax_err, fraction=0.046, pad=0.04)

ax_x_cut = fig.add_subplot(gs[1, 0])
line_x_true, = ax_x_cut.plot(x_ax, data['T_true'][z_center, y_center, :], 'k-', label='True', lw=2)
line_x_pred, = ax_x_cut.plot(x_ax, data['T_pred'][z_center, y_center, :], 'r--', label='PINN', lw=2)
ax_x_cut.set_title("X-Axis Profile")
ax_x_cut.legend()
ax_x_cut.grid(True)

ax_y_cut = fig.add_subplot(gs[1, 1])
line_y_true, = ax_y_cut.plot(y_ax, data['T_true'][z_center, :, x_center], 'k-', label='True', lw=2)
line_y_pred, = ax_y_cut.plot(y_ax, data['T_pred'][z_center, :, x_center], 'r--', label='PINN', lw=2)
ax_y_cut.set_title("Y-Axis Profile")
ax_y_cut.grid(True)

ax_z_cut = fig.add_subplot(gs[1, 2:4])
line_z_true, = ax_z_cut.plot(Z_ARRAY_SMOOTH, data['T_true'][:, y_center, x_center], 'k-', label='True', lw=2)
line_z_pred, = ax_z_cut.plot(Z_ARRAY_SMOOTH, data['T_pred'][:, y_center, x_center], 'r--', label='PINN', lw=2)
vl_z = ax_z_cut.axvline(Z_ARRAY_SMOOTH[z_center], color='magenta', linestyle=':', lw=2, label='Current Z Plane')

def draw_z_backgrounds(ax, pat):
    c = PATTERN_CONFIGS[pat]
    ax.axvspan(0, c['b1'], color='lightgray', alpha=0.4, label=f'Bot Glass (0-{c["b1"]:.0f} µm)')
    ax.axvspan(c['b1'], c['b2'], color='lightblue', alpha=0.4, label=f'Fluid ({c["b1"]:.0f}-{c["b2"]:.0f} µm)')
    ax.axvspan(c['b2'], THICKNESS_Z_UM, color='lightgray', alpha=0.6, label=f'Top Glass ({c["b2"]:.0f}-{THICKNESS_Z_UM:.0f} µm)')

draw_z_backgrounds(ax_z_cut, active_pat)
ax_z_cut.set_title("Z-Axis Thermodynamic Profile")
ax_z_cut.set_xlabel("Depth (µm)")
ax_z_cut.set_ylabel("Temperature Rise (K)")
ax_z_cut.legend(loc='upper right')
ax_z_cut.grid(True)

# --- Text Report ---
ax_report = fig.add_subplot(gs[:, 4])
ax_report.axis('off')

def get_report_text(pat, current_z, z_idx):
    c_data = results_db[pat]
    cfg = PATTERN_CONFIGS[pat]
    max_t_view = np.max(c_data['T_true'][z_idx])
    peak_t_true = np.max(c_data['T_true'])
    peak_t_pinn = np.max(c_data['T_pred'])
    peak_phase = np.max(c_data['phase'])
    
    return (
        f"--- ACTIVE SCENARIO ---\n"
        f"Pattern: {pat.upper()}\n"
        f"Desc: {cfg['desc']}\n"
        f"Heating Power: {P_PUMP_W*1000:.1f} mW\n"
        f"Particle Array Z: {cfg['z0']:.1f} µm\n"
        f"Current View Z: {current_z:.2f} µm\n\n"
        
        f"--- DYNAMIC CHAMBER ---\n"
        f"Top Glass: {THICKNESS_Z_UM - cfg['b2']:.1f} µm\n"
        f"Mid Fluid: {cfg['b2'] - cfg['b1']:.1f} µm\n"
        f"Bot Glass: {cfg['b1']:.1f} µm\n\n"

        f"--- DOMAIN PREDICTIONS ---\n"
        f"Peak Phase Diff : {peak_phase:.2f} rad\n"
        f"Peak True Temp  : {peak_t_true:.1f} K\n"
        f"Peak PINN Temp  : {peak_t_pinn:.1f} K\n"
        f"View True Temp  : {max_t_view:.1f} K\n\n"
        
        f"--- LOCALIZED THERMAL Ks ---\n"
        f"Top (Glass): k={MATERIALS_DB['glass']['k']}\n"
        f"Mid (Fluid): k={cfg['k_mid']}\n"
        f"Bot (Glass): k={MATERIALS_DB['glass']['k']}\n"
    )

report_text_obj = ax_report.text(
    0.05, 0.95, get_report_text(active_pat, Z_ARRAY_SMOOTH[z_center], z_center), 
    transform=ax_report.transAxes, fontsize=11, fontfamily='monospace', 
    verticalalignment='top', horizontalalignment='left',
    bbox=dict(boxstyle="round,pad=1", facecolor="#f8f9fa", edgecolor="lightgray", lw=1.5)
)

# =========================================================================
# --- INTERACTIVE CONTROLS ---
# =========================================================================
axcolor = 'lightgoldenrodyellow'

ax_z = plt.axes([0.15, 0.12, 0.60, 0.02], facecolor=axcolor)
ax_y = plt.axes([0.15, 0.08, 0.60, 0.02], facecolor=axcolor)
ax_x = plt.axes([0.15, 0.04, 0.60, 0.02], facecolor=axcolor)

slider_z = Slider(ax_z, 'Z-Depth (µm)', 0, NZ-1, valinit=z_center, valstep=1)
slider_y = Slider(ax_y, 'Y-Slice', 0, NY-1, valinit=y_center, valstep=1)
slider_x = Slider(ax_x, 'X-Slice', 0, NX-1, valinit=x_center, valstep=1)
slider_z.valtext.set_text(f"{Z_ARRAY_SMOOTH[z_center]:.1f}")

ax_radio = plt.axes([0.80, 0.04, 0.15, 0.10], facecolor='white')
ax_radio.set_title("Swap Pattern Dynamics", weight='bold', pad=10)
radio_pat = RadioButtons(ax_radio, PATTERNS, active=0)

def update(val=None):
    z_idx, y_idx, x_idx = int(slider_z.val), int(slider_y.val), int(slider_x.val)
    current_z = Z_ARRAY_SMOOTH[z_idx]
    
    slider_z.valtext.set_text(f"{current_z:.1f}")
    c_data = results_db[active_pat]
    report_text_obj.set_text(get_report_text(active_pat, current_z, z_idx))
    
    vmax_new = max(np.max(c_data['T_true'][z_idx]), np.max(c_data['T_pred'][z_idx])) + 1e-5
    
    im_true.set_data(c_data['T_true'][z_idx])
    im_true.set_clim(vmin=0, vmax=vmax_new)
    im_pred.set_data(c_data['T_pred'][z_idx])
    im_pred.set_clim(vmin=0, vmax=vmax_new)
    im_err.set_data(c_data['error'][z_idx])
    
    hl_true.set_ydata([y_ax[y_idx], y_ax[y_idx]])
    vl_true.set_xdata([x_ax[x_idx], x_ax[x_idx]])
    hl_pred.set_ydata([y_ax[y_idx], y_ax[y_idx]])
    vl_pred.set_xdata([x_ax[x_idx], x_ax[x_idx]])
    
    line_x_true.set_ydata(c_data['T_true'][z_idx, y_idx, :])
    line_x_pred.set_ydata(c_data['T_pred'][z_idx, y_idx, :])
    line_y_true.set_ydata(c_data['T_true'][z_idx, :, x_idx])
    line_y_pred.set_ydata(c_data['T_pred'][z_idx, :, x_idx])
    line_z_true.set_ydata(c_data['T_true'][:, y_idx, x_idx])
    line_z_pred.set_ydata(c_data['T_pred'][:, y_idx, x_idx])
    vl_z.set_xdata([current_z, current_z])
    
    max_x = max(c_data['T_true'][z_idx, y_idx, :].max(), c_data['T_pred'][z_idx, y_idx, :].max()) * 1.1 + 0.1
    max_y = max(c_data['T_true'][z_idx, :, x_idx].max(), c_data['T_pred'][z_idx, :, x_idx].max()) * 1.1 + 0.1
    max_z = max(c_data['T_true'][:, y_idx, x_idx].max(), c_data['T_pred'][:, y_idx, x_idx].max()) * 1.1 + 0.1
    
    ax_x_cut.set_ylim(-0.1, max_x)
    ax_y_cut.set_ylim(-0.1, max_y)
    ax_z_cut.set_ylim(-0.1, max_z)
    
    fig.canvas.draw_idle()

def swap_pattern(label):
    global active_pat, line_z_true, line_z_pred, vl_z
    active_pat = label
    c_data = results_db[active_pat]
    cfg = PATTERN_CONFIGS[active_pat]
    
    # Update Phase Img
    im_phase.set_data(c_data['phase'])
    im_phase.set_clim(vmin=c_data['phase'].min(), vmax=c_data['phase'].max())
    
    # Snap Z-Slider to the new Source location automatically
    new_z_idx = np.argmin(np.abs(Z_ARRAY_SMOOTH - cfg['z0']))
    slider_z.set_val(new_z_idx)
    
    # Redraw Dynamic Z Boundaries
    ax_z_cut.clear()
    draw_z_backgrounds(ax_z_cut, active_pat)
    line_z_true, = ax_z_cut.plot(Z_ARRAY_SMOOTH, c_data['T_true'][:, int(slider_y.val), int(slider_x.val)], 'k-', label='True', lw=2)
    line_z_pred, = ax_z_cut.plot(Z_ARRAY_SMOOTH, c_data['T_pred'][:, int(slider_y.val), int(slider_x.val)], 'r--', label='PINN', lw=2)
    vl_z = ax_z_cut.axvline(Z_ARRAY_SMOOTH[new_z_idx], color='magenta', linestyle=':', lw=2, label='Current Z Plane')
    ax_z_cut.set_title("Z-Axis Thermodynamic Profile")
    ax_z_cut.set_xlabel("Depth (µm)")
    ax_z_cut.set_ylabel("Temperature Rise (K)")
    ax_z_cut.legend(loc='upper right')
    ax_z_cut.grid(True)
    
    update(None)

slider_z.on_changed(update)
slider_y.on_changed(update)
slider_x.on_changed(update)
radio_pat.on_clicked(swap_pattern)

plt.show()

In [ ]:
%matplotlib qt
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, RadioButtons
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from torch.amp import autocast

# Make sure PGI_Hnet.py is saved with the latest architecture before running!
from PGI_Hnet import PhysicsGuidedInversionUNet
from nanothermal_3D_lib import Native3DTQSISolver, MATERIALS_DB

VERSION = input("Enter the trained model version (e.g., 7.8): ").strip()
MODEL_PATH = rf"D:\kushagra\experimental data\simulation data loader\PGI_Hnet checkpoints\pinn_latest_V{VERSION}.pth"

# =========================================================================
# --- UPDATED PHYSICS PARAMETERS ---
# =========================================================================
NX, NY, NZ_MODEL = 128, 128, 32
WINDOW_XY_UM = 12.8  

THICKNESS_Z_UM = 200.0 
B1_UM = 50.0   
CENTRAL_LAYER_THICKNESS_UM = 100.0  
B2_UM = B1_UM + CENTRAL_LAYER_THICKNESS_UM 
Z_SOURCE_UM = B1_UM + (CENTRAL_LAYER_THICKNESS_UM / 2.0) 

PIXEL_SIZE_UM = WINDOW_XY_UM / NX
Z_ARRAY_UM = np.linspace(0, THICKNESS_Z_UM, NZ_MODEL)

P_PUMP_W = 15.0e-3  
LAMBDA_PUMP_NM = 532.0
LAMBDA_PROBE_NM = 457.0  

LAYER_MATERIALS = ['glass', 'water', 'glass']
CUSTOM_WATER_K = 0.6  
MATERIALS_DB['water']['k'] = CUSTOM_WATER_K

ARRAY_ROWS = 5          
ARRAY_COLS = 5          
PITCH_X_UM = 1.5        
PITCH_Y_UM = 1.5        
NP_RADIUS_NM = 40.0     
NP_MATERIAL = 'gold'    
PATTERNS = ['Grid', 'Single', 'Line', 'Cross', 'Ring']
# =========================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[*] Loading Model Version {VERSION} onto {device}...")
model = PhysicsGuidedInversionUNet(bf=8).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True), strict=False)
model.eval()

def compute_case(pattern_type):
    print(f"    -> Simulating '{pattern_type}' pattern geometry...")
    solver = Native3DTQSISolver(
        pixel_size_um=PIXEL_SIZE_UM, z_array_um=Z_ARRAY_UM,
        layer_materials=LAYER_MATERIALS, boundaries_um=[B1_UM, B2_UM],
        a_m=50e-9, probe_nm=LAMBDA_PROBE_NM
    )
    
    n_water = MATERIALS_DB['water']['n']
    sigma_abs = Native3DTQSISolver.compute_optical_properties(NP_RADIUS_NM, LAMBDA_PUMP_NM, MATERIALS_DB[NP_MATERIAL]['n'], n_water)
    
    coords = []
    if pattern_type == 'Grid':
        start_x = - (ARRAY_COLS - 1) * PITCH_X_UM / 2.0
        start_y = - (ARRAY_ROWS - 1) * PITCH_Y_UM / 2.0
        for r in range(ARRAY_ROWS):
            for c in range(ARRAY_COLS):
                coords.append((start_x + c * PITCH_X_UM, start_y + r * PITCH_Y_UM))
    elif pattern_type == 'Single':
        coords.append((0.0, 0.0))
    elif pattern_type == 'Line':
        start_x = - (ARRAY_COLS - 1) * PITCH_X_UM / 2.0
        for c in range(ARRAY_COLS): coords.append((start_x + c * PITCH_X_UM, 0.0))
    elif pattern_type == 'Cross':
        start_y = - (ARRAY_ROWS - 1) * PITCH_Y_UM / 2.0
        for r in range(ARRAY_ROWS): coords.append((0.0, start_y + r * PITCH_Y_UM))
        start_x = - (ARRAY_COLS - 1) * PITCH_X_UM / 2.0
        for c in range(ARRAY_COLS):
            x = start_x + c * PITCH_X_UM
            if abs(x) > 1e-5: coords.append((x, 0.0))
    elif pattern_type == 'Ring':
        num_particles = 8
        radius = PITCH_X_UM * 1.5
        for i in range(num_particles):
            angle = i * (2 * np.pi / num_particles)
            coords.append((radius * np.cos(angle), radius * np.sin(angle)))

    coords = [(x, y) for x, y in coords if abs(x) < WINDOW_XY_UM/2 and abs(y) < WINDOW_XY_UM/2]
    particles_array = [{'x0': x, 'y0': y, 'radius_nm': NP_RADIUS_NM, 'material': NP_MATERIAL, 'sigma_abs': sigma_abs} for x, y in coords]
    
    _, T_true_3d, source_3d = solver.generate_synthetic_phase(NY, NX, particles_array, P_PUMP_W, LAMBDA_PUMP_NM, z0_um=Z_SOURCE_UM)
    
    T_tensor = torch.tensor(T_true_3d, dtype=solver.dtype, device=solver.device)
    dn_3d = torch.zeros_like(T_tensor)
    
    m_bot = (solver.z_array < solver.b1).view(-1,1,1).expand_as(dn_3d)
    m_mid = ((solver.z_array >= solver.b1) & (solver.z_array <= solver.b2)).view(-1,1,1).expand_as(dn_3d)
    m_top = (solver.z_array > solver.b2).view(-1,1,1).expand_as(dn_3d)
    
    dn_3d[m_bot] = solver._eval_dn(T_tensor, solver.beta_bot)[m_bot]
    dn_3d[m_mid] = solver._eval_dn(T_tensor, solver.beta_mid)[m_mid]
    dn_3d[m_top] = solver._eval_dn(T_tensor, solver.beta_top)[m_top] 
    
    opd_2d = torch.sum(dn_3d * solver.dz, dim=0)
    phase_2d = (opd_2d * (2 * np.pi / solver.lambda_probe)).cpu().numpy()
    _, T_inv_3d = solver.invert_tqsi_3d(phase_2d, z0_um=Z_SOURCE_UM, iterations=3)
    
    k_vol = np.zeros((NZ_MODEL, NY, NX), dtype=np.float32)
    beta_vol = np.zeros((NZ_MODEL, NY, NX), dtype=np.float32)
    k_vol[Z_ARRAY_UM < B1_UM] = MATERIALS_DB['glass']['k']
    k_vol[(Z_ARRAY_UM >= B1_UM) & (Z_ARRAY_UM <= B2_UM)] = MATERIALS_DB['water']['k']
    k_vol[Z_ARRAY_UM > B2_UM] = MATERIALS_DB['glass']['k']
    
    beta_vol[Z_ARRAY_UM < B1_UM] = MATERIALS_DB['glass']['beta'][0]
    beta_vol[(Z_ARRAY_UM >= B1_UM) & (Z_ARRAY_UM <= B2_UM)] = MATERIALS_DB['water']['beta'][0]
    beta_vol[Z_ARRAY_UM > B2_UM] = MATERIALS_DB['glass']['beta'][0]

    with torch.no_grad():
        with autocast('cuda'):
            phase_t = torch.tensor(phase_2d, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            t_inv_t = torch.tensor(T_inv_3d, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            k_t     = torch.tensor(k_vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            beta_t  = torch.tensor(beta_vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            
            # [FIXED]: Unpack the tuple (T_pred_tensor, dynamic_peak) returned by model()
            T_pred_tensor, _ = model(phase_t, t_inv_t, k_t, beta_t)
            T_pred_3d = T_pred_tensor.squeeze().cpu().numpy()

    error_map = np.abs(T_pred_3d - T_true_3d) / (np.max(T_true_3d) + 1e-5) * 100.0

    SMOOTH_NZ = 96 
    smooth_z_array = np.linspace(0, THICKNESS_Z_UM, SMOOTH_NZ)
    T_true_3d = interp1d(Z_ARRAY_UM, T_true_3d, axis=0, kind='cubic')(smooth_z_array)
    T_pred_3d = interp1d(Z_ARRAY_UM, T_pred_3d, axis=0, kind='cubic')(smooth_z_array)
    error_map = interp1d(Z_ARRAY_UM, error_map, axis=0, kind='linear')(smooth_z_array)

    return {
        'phase': phase_2d, 'T_true': T_true_3d, 'T_pred': T_pred_3d, 
        'error': error_map, 'source': source_3d, 'sigma': sigma_abs, 'z_arr': smooth_z_array
    }

print("\n[*] PRE-COMPUTING ALL PATTERN SCENARIOS (This will take ~10 seconds)...")
results_db = {pat: compute_case(pat) for pat in PATTERNS}
print("[*] Pre-computation complete! Launching interactive dashboard.")

# =========================================================================
# --- GUI SETUP ---
# =========================================================================
active_pat = 'Grid'
data = results_db[active_pat]
Z_ARRAY_SMOOTH = data['z_arr']
NZ = len(Z_ARRAY_SMOOTH)

y_ax = np.linspace(-NY//2, NY//2 - 1, NY) * PIXEL_SIZE_UM
x_ax = np.linspace(-NX//2, NX//2 - 1, NX) * PIXEL_SIZE_UM

z_center = np.argmin(np.abs(Z_ARRAY_SMOOTH - Z_SOURCE_UM)) 
y_center, x_center = NY // 2, NX // 2

fig = plt.figure(figsize=(24, 12)) 
gs = GridSpec(2, 5, figure=fig, height_ratios=[1.5, 1], width_ratios=[1, 1, 1, 1, 1.2], 
              bottom=0.25, top=0.92, hspace=0.3, wspace=0.3) 

main_title = fig.suptitle(f"Physics-Guided Inversion UNet Dashboard V{VERSION} | Intensive Heating Profile", fontsize=18, weight='bold', y=0.98)

ax_phase = fig.add_subplot(gs[0, 0])
im_phase = ax_phase.imshow(data['phase'], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='viridis')
ax_phase.set_title("Input 2D Phase Map (rad)")
cb_phase = plt.colorbar(im_phase, ax=ax_phase, fraction=0.046, pad=0.04)

vmax_shared = max(np.max(data['T_true'][z_center]), np.max(data['T_pred'][z_center])) + 1e-5

ax_true = fig.add_subplot(gs[0, 1])
im_true = ax_true.imshow(data['T_true'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='hot', vmin=0, vmax=vmax_shared)
ax_true.set_title(r"Ground Truth $T_{true}(x,y)$")
cb_true = plt.colorbar(im_true, ax=ax_true, fraction=0.046, pad=0.04)
hl_true = ax_true.axhline(y_ax[y_center], color='cyan', linestyle='--', alpha=0.7)
vl_true = ax_true.axvline(x_ax[x_center], color='lime', linestyle='--', alpha=0.7)

ax_pred = fig.add_subplot(gs[0, 2])
im_pred = ax_pred.imshow(data['T_pred'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='hot', vmin=0, vmax=vmax_shared)
ax_pred.set_title(r"PINN Prediction $T_{pred}(x,y)$")
cb_pred = plt.colorbar(im_pred, ax=ax_pred, fraction=0.046, pad=0.04)
hl_pred = ax_pred.axhline(y_ax[y_center], color='cyan', linestyle='--', alpha=0.7)
vl_pred = ax_pred.axvline(x_ax[x_center], color='lime', linestyle='--', alpha=0.7)

ax_err = fig.add_subplot(gs[0, 3])
im_err = ax_err.imshow(data['error'][z_center], extent=[x_ax[0], x_ax[-1], y_ax[0], y_ax[-1]], origin='lower', cmap='Reds', vmin=0, vmax=20)
ax_err.set_title("Absolute Error (%)")
cb_err = plt.colorbar(im_err, ax=ax_err, fraction=0.046, pad=0.04)

ax_x_cut = fig.add_subplot(gs[1, 0])
line_x_true, = ax_x_cut.plot(x_ax, data['T_true'][z_center, y_center, :], 'k-', label='True', lw=2)
line_x_pred, = ax_x_cut.plot(x_ax, data['T_pred'][z_center, y_center, :], 'r--', label='PINN', lw=2)
ax_x_cut.set_title("X-Axis Profile")
ax_x_cut.legend()
ax_x_cut.grid(True)

ax_y_cut = fig.add_subplot(gs[1, 1])
line_y_true, = ax_y_cut.plot(y_ax, data['T_true'][z_center, :, x_center], 'k-', label='True', lw=2)
line_y_pred, = ax_y_cut.plot(y_ax, data['T_pred'][z_center, :, x_center], 'r--', label='PINN', lw=2)
ax_y_cut.set_title("Y-Axis Profile")
ax_y_cut.grid(True)

ax_z_cut = fig.add_subplot(gs[1, 2:4])
ax_z_cut.axvspan(0, B1_UM, color='lightgray', alpha=0.4, label=f'Bot Glass (0-{B1_UM:.0f} µm)')
ax_z_cut.axvspan(B1_UM, B2_UM, color='lightblue', alpha=0.4, label=f'Water ({B1_UM:.0f}-{B2_UM:.0f} µm)')
ax_z_cut.axvspan(B2_UM, THICKNESS_Z_UM, color='lightgray', alpha=0.6, label=f'Top Glass ({B2_UM:.0f}-{THICKNESS_Z_UM:.0f} µm)')

line_z_true, = ax_z_cut.plot(Z_ARRAY_SMOOTH, data['T_true'][:, y_center, x_center], 'k-', label='True', lw=2)
line_z_pred, = ax_z_cut.plot(Z_ARRAY_SMOOTH, data['T_pred'][:, y_center, x_center], 'r--', label='PINN', lw=2)
vl_z = ax_z_cut.axvline(Z_ARRAY_SMOOTH[z_center], color='magenta', linestyle=':', lw=2, label='Current Z Plane')

ax_z_cut.set_title("Z-Axis Thermodynamic Profile")
ax_z_cut.set_xlabel("Depth (µm)")
ax_z_cut.set_ylabel("Temperature Rise (K)")
ax_z_cut.legend(loc='upper right')
ax_z_cut.grid(True)

# --- Text Report ---
ax_report = fig.add_subplot(gs[:, 4])
ax_report.axis('off')

def get_report_text(pat, current_z, z_idx):
    c_data = results_db[pat]
    max_t_view = np.max(c_data['T_true'][z_idx])
    peak_t_true = np.max(c_data['T_true'])
    peak_t_pinn = np.max(c_data['T_pred'])
    peak_phase = np.max(c_data['phase'])
    
    return (
        f"--- SIMULATION PARAMETERS ---\n"
        f"Resolution: {NX}x{NY} px (dx={PIXEL_SIZE_UM*1000:.1f} nm)\n"
        f"Active Pattern: {pat.upper()} | Pitch: {PITCH_X_UM}x{PITCH_Y_UM} µm\n"
        f"Particle Radius (a): {NP_RADIUS_NM} nm\n"
        f"Heating Power: {P_PUMP_W*1000:.1f} mW\n"
        f"Pump λ: {LAMBDA_PUMP_NM:.1f} nm\n"
        f"Probe λ: {LAMBDA_PROBE_NM:.1f} nm\n"
        f"Calculated σ_abs: {c_data['sigma']:.2e} µm²\n"
        f"Current View Z: {current_z:.2f} µm\n\n"
        
        f"--- CHAMBER GEOMETRY ---\n"
        f"Total Height (Z): {THICKNESS_Z_UM:.1f} µm\n"
        f"Top Glass Layer : {THICKNESS_Z_UM - B2_UM:.1f} µm\n"
        f"Mid Water Layer : {B2_UM - B1_UM:.1f} µm\n"
        f"Bot Glass Layer : {B1_UM:.1f} µm\n\n"

        f"--- MAXIMUM PREDICTIONS ---\n"
        f"Peak Phase Diff : {peak_phase:.2f} rad\n"
        f"Peak True Temp  : {peak_t_true:.1f} K\n"
        f"Peak PINN Temp  : {peak_t_pinn:.1f} K\n"
        f"View True Temp  : {max_t_view:.1f} K\n\n"
        
        f"--- THERMAL DOMAIN (W/m·K) ---\n"
        f"Layer 3 (Glass): k={MATERIALS_DB['glass']['k']}, β={MATERIALS_DB['glass']['beta'][0]:.2e}\n"
        f"Layer 2 (Water): k={CUSTOM_WATER_K}, β={MATERIALS_DB['water']['beta'][0]:.2e}\n"
        f"Layer 1 (Glass): k={MATERIALS_DB['glass']['k']}, β={MATERIALS_DB['glass']['beta'][0]:.2e}\n"
    )

report_text_obj = ax_report.text(
    0.05, 0.95, get_report_text(active_pat, Z_ARRAY_SMOOTH[z_center], z_center), 
    transform=ax_report.transAxes, fontsize=12, fontfamily='monospace', 
    verticalalignment='top', horizontalalignment='left',
    bbox=dict(boxstyle="round,pad=1", facecolor="#f8f9fa", edgecolor="lightgray", lw=1.5)
)

# =========================================================================
# --- INTERACTIVE CONTROLS (SLIDERS & RADIO BUTTONS) ---
# =========================================================================
axcolor = 'lightgoldenrodyellow'

ax_z = plt.axes([0.15, 0.12, 0.60, 0.02], facecolor=axcolor)
ax_y = plt.axes([0.15, 0.08, 0.60, 0.02], facecolor=axcolor)
ax_x = plt.axes([0.15, 0.04, 0.60, 0.02], facecolor=axcolor)

slider_z = Slider(ax_z, 'Z-Depth (µm)', 0, NZ-1, valinit=z_center, valstep=1)
slider_y = Slider(ax_y, 'Y-Slice', 0, NY-1, valinit=y_center, valstep=1)
slider_x = Slider(ax_x, 'X-Slice', 0, NX-1, valinit=x_center, valstep=1)

slider_z.valtext.set_text(f"{Z_ARRAY_SMOOTH[z_center]:.1f}")

ax_radio = plt.axes([0.80, 0.04, 0.15, 0.10], facecolor='white')
ax_radio.set_title("Swap Pattern Dynamics", weight='bold', pad=10)
radio_pat = RadioButtons(ax_radio, PATTERNS, active=0)

def update(val):
    z_idx, y_idx, x_idx = int(slider_z.val), int(slider_y.val), int(slider_x.val)
    current_z = Z_ARRAY_SMOOTH[z_idx]
    
    slider_z.valtext.set_text(f"{current_z:.1f}")
    
    c_data = results_db[active_pat]
    report_text_obj.set_text(get_report_text(active_pat, current_z, z_idx))
    
    vmax_new = max(np.max(c_data['T_true'][z_idx]), np.max(c_data['T_pred'][z_idx])) + 1e-5
    
    im_true.set_data(c_data['T_true'][z_idx])
    im_true.set_clim(vmin=0, vmax=vmax_new)
    
    im_pred.set_data(c_data['T_pred'][z_idx])
    im_pred.set_clim(vmin=0, vmax=vmax_new)
    
    im_err.set_data(c_data['error'][z_idx])
    
    hl_true.set_ydata([y_ax[y_idx], y_ax[y_idx]])
    vl_true.set_xdata([x_ax[x_idx], x_ax[x_idx]])
    hl_pred.set_ydata([y_ax[y_idx], y_ax[y_idx]])
    vl_pred.set_xdata([x_ax[x_idx], x_ax[x_idx]])
    
    line_x_true.set_ydata(c_data['T_true'][z_idx, y_idx, :])
    line_x_pred.set_ydata(c_data['T_pred'][z_idx, y_idx, :])
    
    line_y_true.set_ydata(c_data['T_true'][z_idx, :, x_idx])
    line_y_pred.set_ydata(c_data['T_pred'][z_idx, :, x_idx])
    
    line_z_true.set_ydata(c_data['T_true'][:, y_idx, x_idx])
    line_z_pred.set_ydata(c_data['T_pred'][:, y_idx, x_idx])
    vl_z.set_xdata([current_z, current_z])
    
    max_x = max(c_data['T_true'][z_idx, y_idx, :].max(), c_data['T_pred'][z_idx, y_idx, :].max()) * 1.1 + 0.1
    max_y = max(c_data['T_true'][z_idx, :, x_idx].max(), c_data['T_pred'][z_idx, :, x_idx].max()) * 1.1 + 0.1
    max_z = max(c_data['T_true'][:, y_idx, x_idx].max(), c_data['T_pred'][:, y_idx, x_idx].max()) * 1.1 + 0.1
    
    ax_x_cut.set_ylim(-0.1, max_x)
    ax_y_cut.set_ylim(-0.1, max_y)
    ax_z_cut.set_ylim(-0.1, max_z)
    
    fig.canvas.draw_idle()

def swap_pattern(label):
    global active_pat
    active_pat = label
    
    c_data = results_db[active_pat]
    im_phase.set_data(c_data['phase'])
    im_phase.set_clim(vmin=c_data['phase'].min(), vmax=c_data['phase'].max())
    
    update(None)

slider_z.on_changed(update)
slider_y.on_changed(update)
slider_x.on_changed(update)
radio_pat.on_clicked(swap_pattern)

plt.show()

